In [29]:
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import re
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
from sklearn.metrics import classification_report, f1_score
import torch.utils.data as data_utils
import torch.optim as optim
import os
import joblib
from catboost import CatBoostClassifier

In [47]:
class Tokeniser():
    def __init__(self, config={}):
        self.vocab_idtotoken = {}
        self.vocab_tokentoid = {}

    def generate_vocab(self, text, create=True, way='vocab.json', use_saved=True):
        if os.path.exists(way) and use_saved:
            try:
                with open(way) as f:
                    data = json.load(f)
                self.vocab_idtotoken = data['vocab_idtotoken']
                self.vocab_tokentoid = data['vocab_tokentoid']
                if len(self.vocab_idtotoken) == len(self.vocab_tokentoid) and len(self.vocab_tokentoid)>1000:
                    return None
            except:
                pass
        special_tokens = ['<EOS>', '<BOS>', '<UNK>']
        alpha_en = 'qazxswedcvfrtgbnhyujmkiolp'
        num = '0123456789'
        alpha_ru = 'йфячыцувсмакепитнргоьблшщдюжзхэъ'
        tokens = sorted(set(text+alpha_ru+alpha_ru.upper()+num+alpha_en+alpha_en.upper()))
        tokens_intext = ['##'+ i for i in tokens]
        tokens.extend(tokens_intext)
        tokens.extend(special_tokens)
        tokens.extend(self.get_tokens(2, text, 500))
        tokens.extend(self.get_tokens(3, text, 400))
        tokens.extend(self.get_tokens(4, text, 300))
        tokens.extend(self.get_tokens(5, text, 200))
        tokens.extend(self.get_tokens(6, text, 100))
        tokens.extend(self.get_tokens(7, text, 100))
        tokens.extend(self.get_tokens(8, text, 100))
        tokens.extend(self.get_tokens(9, text, 100))
        for id in range(len(tokens)):
            self.vocab_idtotoken[id] = tokens[id]
            self.vocab_tokentoid[tokens[id]] = id
        if create:
            with open(way, 'w') as f:
                json.dump({
                    'vocab_idtotoken':self.vocab_idtotoken,
                    'vocab_tokentoid':self.vocab_tokentoid
                          }, f)
            with open(way) as f:
                data = json.load(f)
            self.vocab_idtotoken = data['vocab_idtotoken']
            self.vocab_tokentoid = data['vocab_tokentoid']

    def __len__(self):
        return len(self.vocab_idtotoken)

    def get_tokens(self, length, text, count_times=30):
        text_to_tokens = re.sub('[^\w@] ', '', text).replace('BOS', '').replace('EOS', '')
        sumbols_count = {}
        for word in tqdm(text_to_tokens.split(' '), desc=f'Generate {length} sumbols vocab'):
            for id in range(len(word)//length):
                token = word[length*id:length*(id+1)]
                if length*id != 0:
                    token = '##'+token
                if token in sumbols_count:
                    sumbols_count[token] += 1
                else:
                    sumbols_count[token] = 1
        sumbols_count = sorted(sumbols_count.items(), key=lambda x: x[1], reverse=True)
        sumbols_count = [i[0] for i in sumbols_count if i[1]>=count_times]
        return sumbols_count
        
    def tokenize(self, text):
        tokenized_text = []
        words = text.split(' ')
        for word in words:
            if not word:
                continue
            word_sub = word
            word_result = []
            while word.replace(' ', '') != '':
                replace_token = '_'
                if replace_token in word:
                    replacevocab = '_*-/=+-—~Æ¯‘„ÎŸ›$#|ˆ{'
                    for i in replacevocab:
                        if i not in word:
                            replace_token = i
                            break
                start = min(9, len(word))
                found = False
                for count in range(start, 0, -1):
                    for i in range(len(word) - count + 1):
                        token = word[i:i + count]
                        if token == ' ':
                            continue
                        id_real = word_sub.find(token)
                        if id_real != 0:
                            token_tosearch = '##'+token
                        else:
                            token_tosearch = token
                        token_id = self.vocab_tokentoid.get(token_tosearch, '')
                        if isinstance(token_id, str) and token_id.isdigit():
                            token_id = int(token_id)
                        if isinstance(token_id, int):
                            word_result.append([int(token_id), id_real])
                            # print(f'Token is |{token}| in the |{word}| from {i} to {i + count}')
                            # print(f'Sub word is |{word_sub}| start token with {id_real}')
                            word_sub = word_sub.replace(token, replace_token*len(token), 1)
                            word = word[:i] + ' ' + word[i + count:]
                            found = True
                            break
                    if found:
                        break
                if not found:
                    token_id = self.vocab_tokentoid.get('<UNK>', None)
                    if token_id is not None:
                        id_real = word_sub.find(token)
                        word_result.append(int(token_id))
                        word_sub = word_sub.replace(token, replace_token*len(token), 1)
                        word = word[1:]
                    else:
                        raise ValueError(f"Неизвестный токен: {token}")
            word_result = [i[0] for i in sorted(word_result, key=lambda x: x[1])]
            tokenized_text.extend(word_result)
        return tokenized_text

    def detokenize(self, tokenized_text):
        text = []
        for id in tqdm(tokenized_text, desc='Detokenising data'):
            token = self.vocab_idtotoken[str(id)]
            text.append(token)
        return text

    def totext(self, tokenized_text):
        text = ''
        tokens = self.detokenize(tokenized_text)
        now = tokens[0]
        for id in range(1, len(tokens)):
            next = tokens[id]
            if '##' in next:
                text+=now.replace('##', '')
            else:
                text+=now.replace('##', '')+' '
            now = next
        text += now.replace('##', '')
        return text

In [48]:
data = pd.read_csv('/kaggle/input/rubsova-twit/normed.csv')
text = ' <EOS> <BOS> '.join([str(i) for i in data['clean_text'].to_list()])
text = re.sub(' +', ' ', re.sub('([^\w<>@])', r' \1 ', text)).strip()

In [49]:
data['clean_text'] = data['clean_text'].astype(str).apply(lambda x: re.sub(' +', ' ', re.sub(r'([^\w<>@])', r' \1 ', x)).strip())

In [50]:
print(text[:1000])

работа полный пиддес каждый закрытие месяц свихнуться <EOS> <BOS> коллега сидеть рубиться изз долбать винд не мочь <EOS> <BOS> 4 говорить обещаной год ждать <EOS> <BOS> желать хороший полёт удачный посадкия быть очень сильно скучать 3 <EOS> <BOS> обновить какимтый леший не работать простоплеер <EOS> <BOS> котёнок вчера носик разбить плакать расстраиваться <EOS> <BOS> 55 заслать опять затихариться прямо физически страдать долго молчать <EOS> <BOS> вообще не болеть не выздоравливать <EOS> <BOS> микрофраза учиться срать кирпич режим нонстоп <EOS> <BOS> хотеть ты помириться сука гордый никогда не сделать <EOS> <BOS> 33 33 ебета какой фоткия твой молчуя вообще знать тп выглядеть <EOS> <BOS> блин начать сниться сон не не сниться сказать каждый день такой это не нравиться <EOS> <BOS> твой место сначала телек купить <EOS> <BOS> 111 плохо бояться значит мск бгр <EOS> <BOS> хотеть электромобиль 9 <EOS> <BOS> скоро увидеть твой зелёный глаз последний <EOS> <BOS> это докатилсяпооранжеветь патриот 

In [51]:
tokeniser = Tokeniser()
tokeniser.generate_vocab(
                        text, 
                        use_saved=False
                        )
len(tokeniser)

Generate 9 sumbols vocab: 100%|██████████| 1634056/1634056 [00:00<00:00, 1643314.15it/s]


8675

In [52]:
tokeniser.tokenize('Это лабораторная работа по NLP')

[95, 288, 109, 1589, 1081, 327, 299, 260, 4721, 269, 27, 156, 160]

In [85]:
tokeniser.detokenize([95, 288, 109, 1589, 1081, 327, 299, 260, 4721, 269, 27, 156, 160])

Detokenising data: 100%|██████████| 13/13 [00:00<00:00, 50486.99it/s]


['Э',
 '##то',
 'л',
 '##або',
 '##рат',
 '##ор',
 '##на',
 '##я',
 'работа',
 'по',
 'N',
 '##L',
 '##P']

In [53]:
tokeniser.totext([95, 288, 109, 1589, 1081, 327, 299, 260, 4721, 269, 27, 156, 160])

Detokenising data: 100%|██████████| 13/13 [00:00<00:00, 90876.59it/s]


'Это лабораторная работа по NLP'

In [54]:
X = [tokeniser.tokenize(i) for i in data['clean_text'].to_list()] 

In [56]:
max_len = 0
for i in X:
    if len(i) > max_len:
        max_len = len(i)

In [63]:
X = np.array([np.pad(seq, (0, max_len - len(seq)), mode='constant', constant_values=0) for seq in X])

In [70]:
y = data['typr'].replace(-1, 0).to_numpy()

In [78]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

X_train = torch.tensor(X_train, dtype=torch.long) 
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [79]:
vocab_size = len(tokeniser)
embedding_dim = 128
hidden_dim = 256
output_dim = len(set(y))

In [80]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x).mean(dim=1)  # Усредняем эмбеддинги слов
        x = self.relu(self.fc(x))
        x = self.out(x)
        return x

In [81]:
model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Обучение модели
epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')

Epoch 1/10, Loss: 0.6997
Epoch 2/10, Loss: 0.7132
Epoch 3/10, Loss: 0.6981
Epoch 4/10, Loss: 0.6940
Epoch 5/10, Loss: 0.7010
Epoch 6/10, Loss: 0.6999
Epoch 7/10, Loss: 0.6942
Epoch 8/10, Loss: 0.6933
Epoch 9/10, Loss: 0.6966
Epoch 10/10, Loss: 0.6972


In [82]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()
    y_test_np = y_test.cpu().numpy()

# Вычисление F1-score
f1 = f1_score(y_test_np, test_preds, average='micro')  # Можно заменить на 'macro' или 'weighted'
print(f'F1-score: {f1:.4f}')

F1-score: 0.5058


In [83]:
model = TextClassifier(vocab_size, embedding_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Обучение модели
epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()
    y_test_np = y_test.cpu().numpy()

# Вычисление F1-score
f1 = f1_score(y_test_np, test_preds, average='micro')  # Можно заменить на 'macro' или 'weighted'
print(f'F1-score: {f1:.4f}')

Epoch 1/10, Loss: 0.7068
Epoch 2/10, Loss: 0.7029
Epoch 3/10, Loss: 0.6997
Epoch 4/10, Loss: 0.6971
Epoch 5/10, Loss: 0.6952
Epoch 6/10, Loss: 0.6940
Epoch 7/10, Loss: 0.6933
Epoch 8/10, Loss: 0.6931
Epoch 9/10, Loss: 0.6932
Epoch 10/10, Loss: 0.6936
F1-score: 0.5053


In [84]:
model = TextClassifier(vocab_size, embedding_dim*2, hidden_dim*2, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Обучение модели
epochs = 10
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    test_preds = torch.argmax(test_outputs, dim=1).cpu().numpy()
    y_test_np = y_test.cpu().numpy()

# Вычисление F1-score
f1 = f1_score(y_test_np, test_preds, average='micro')  # Можно заменить на 'macro' или 'weighted'
print(f'F1-score: {f1:.4f}')

Epoch 1/10, Loss: 0.7187
Epoch 2/10, Loss: 0.7008
Epoch 3/10, Loss: 0.7088
Epoch 4/10, Loss: 0.6980
Epoch 5/10, Loss: 0.6934
Epoch 6/10, Loss: 0.6986
Epoch 7/10, Loss: 0.7004
Epoch 8/10, Loss: 0.6965
Epoch 9/10, Loss: 0.6932
Epoch 10/10, Loss: 0.6942
F1-score: 0.4946
